# Session 2: Dataset Preparation, Tuning & Ensembles

This session covers three critical topics for building production-quality risk prediction systems:

**Part 1: Advanced Dataset Creation**
- Handling class imbalance (rare events)
- Detecting and interpreting feature-target correlations

> **Note:** Patient-level dependencies (one patient = many time points) are addressed in Part 2 via GroupKFold cross-validation.

**Part 2: Hyperparameter Tuning**
- Patient-aware cross-validation (GroupKFold)
- Random hyperparameter search for Random Forest
- XGBoost with built-in `xgb.cv()` and early stopping

**Part 3: Classifier Ensembles**
- Voting ensembles (hard/soft voting)
- Weighted ensembles
- Stacking classifiers

---

## What You'll Learn

1. Create high-quality training datasets for risk prediction
2. Handle the unique challenges of longitudinal patient data
3. Tune hyperparameters using patient-aware cross-validation
4. Combine multiple models into ensembles for improved robustness

---

## Why This Matters

In healthcare ML, **dataset creation is as important as model selection**:
- Poor data preparation leads to models that don't generalize
- Data leakage gives falsely optimistic results
- Proper hyperparameter tuning can significantly boost performance
- Ensembles can improve robustness, though gains over a strong single model are often marginal on tabular data

By the end of this session, you'll have a properly created dataset, tuned models, and a trained ensemble.

---

## Setup & Imports

We import libraries for:
- **Data processing**: pandas, numpy
- **Visualization**: matplotlib, seaborn
- **Models**: RandomForest, XGBoost, LogisticRegression
- **Ensembles**: VotingClassifier, StackingClassifier
- **Imbalance handling**: SMOTE, RandomUnderSampler from imblearn
- **Evaluation**: ROC-AUC, PR-AUC metrics

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src/ xgboost imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.base import clone
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

print("Libraries imported")


In [ ]:
import os

DATA_DIR = os.path.join(REPO_PATH, 'data', 'week_2')
OUTPUT_DIR = os.path.join(REPO_PATH, 'data', 'week_3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"DATA_DIR:   {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---

## Load Data

We load the full classifier training data from Week 2. This contains daily instances for diabetic patients with features computed at each date.

In [ ]:
# Load the full classifier training data
df = pd.read_csv(os.path.join(DATA_DIR, 'classifier_training_data_22_features.csv'), low_memory=False)

print(f"Loaded {len(df):,} instances from {df['patient_id'].nunique():,} patients")

# Define columns
label_col = 'will_have_high_risk_event_next_30d'
patient_col = 'patient_id'
date_col = 'date'

exclude_cols = ['patient_id', 'date']
exclude_patterns = ['label', 'risk', 'target', 'will_have', 'next', 'year_month', 'survival']
feature_cols = [col for col in df.columns
                if col not in exclude_cols
                and not any(pat in col.lower() for pat in exclude_patterns)]

print(f"Features: {len(feature_cols)}")

---

# Part 1: Advanced Dataset Creation

Creating a proper training dataset for patient risk prediction requires careful handling of:

| Challenge | Problem | Solution |
|-----------|---------|----------|
| **Class imbalance** | Adverse events are rare (<5%) | SMOTE, class weights, undersampling |
| **Leakage risk** | Future information leaks into features | Correlation audit, GroupKFold |

We use the full classifier training data from Week 2 and focus on handling class imbalance and verifying data integrity.

## 1.1 Addressing Class Imbalance

High-risk events are rare (typically <5% of data points). This causes problems:
- Models learn to predict "no risk" for everyone
- Achieves high accuracy (95%!) but is completely useless
- Misses the patients we actually need to identify

**Solutions:**
| Method | How It Works | Pros | Cons |
|--------|--------------|------|------|
| **Class weights** | Penalize misclassifying rare class | Simple, no data change | May not be enough |
| **SMOTE** | Create synthetic minority examples | Increases minority samples | May create unrealistic examples |
| **Undersampling** | Reduce majority class | Balanced classes | Loses information |

In [ ]:
# Current class distribution
print("CLASS DISTRIBUTION")
print("=" * 40)
pos_rate = df[label_col].mean()
print(f"Positive (high risk): {df[label_col].sum():,} ({pos_rate*100:.2f}%)")
print(f"Negative (no risk): {(df[label_col]==0).sum():,} ({(1-pos_rate)*100:.2f}%)")
print(f"\nImbalance ratio: 1:{int(1/max(pos_rate, 0.001))}")

In [ ]:
# Dual imputation strategy:
#   X     = median-imputed (for sklearn models: RF, LR, SMOTE, ensembles)
#   X_raw = NaN passthrough (for XGBoost, which handles NaN natively)
X_raw = df[feature_cols].copy()
y = df[label_col].astype(int)
groups = df[patient_col]

# Split FIRST, then impute (fit on train only to prevent leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

X_train_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# Fit imputer on training data only
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=feature_cols,
    index=X_train_raw.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=feature_cols,
    index=X_test_raw.index
)

print(f"Train: {len(X_train):,} ({groups_train.nunique()} patients)")
print(f"Test:  {len(X_test):,} ({groups.iloc[test_idx].nunique()} patients)")
print(f"Train positive rate: {y_train.mean()*100:.2f}%")
print(f"\nImputation: median (train-only) for sklearn models, NaN passthrough for XGBoost")

In [ ]:
# Method 1: Class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Class weights (balanced):")
print(f" Class 0 (no risk): {weight_dict[0]:.3f}")
print(f" Class 1 (high risk): {weight_dict[1]:.3f}")

In [ ]:
# Method 2: SMOTE oversampling (only on training data!)
# Caveat: SMOTE is not patient-group-aware — it interpolates between feature vectors
# from different patients, creating synthetic instances that blend two patients' clinical
# profiles. This is acceptable for demonstration but in production you should consider
# patient-aware resampling or stick with class weights.

# TODO: Apply SMOTE to the training data
#   1. Create a SMOTE instance (random_state=42)
#   2. Call fit_resample(X_train, y_train) to get oversampled data
#   HINT: smote = SMOTE(random_state=42), then smote.fit_resample(...)
smote = None
X_train_smote = None
y_train_smote = None

print("SMOTE Oversampling:")
print(f" Before: {len(X_train):,} ({y_train.sum():,} positive)")
print(f" After: {len(X_train_smote):,} ({y_train_smote.sum():,} positive)")

In [ ]:
# Method 3: Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Random Undersampling:")
print(f" Before: {len(X_train):,} ({y_train.sum():,} positive)")
print(f" After: {len(X_train_under):,} ({y_train_under.sum():,} positive)")

In [ ]:
# Compare all four imbalance handling methods using Random Forest
# TODO: Train 4 separate RandomForestClassifier models (n_estimators=100, random_state=42, n_jobs=-1)
#   and evaluate each with average_precision_score on the test set.
#
#   1. Baseline (no handling): fit on X_train, y_train
#   2. Class weights: fit on X_train, y_train with class_weight='balanced'
#   3. SMOTE: fit on X_train_smote, y_train_smote
#   4. Undersampling: fit on X_train_under, y_train_under
#
#   HINT: For each model, call predict_proba(X_test)[:, 1] then average_precision_score()
results = {}

# Baseline (no handling)
rf_comparison = None  # named rf_comparison to avoid confusion with rf_base in Part 3
results['No handling'] = None

# Class weights
rf_weighted = None
results['Class weights'] = None

# SMOTE
rf_smote = None
results['SMOTE'] = None

# Undersampling
rf_under = None
results['Undersampling'] = None

print("\nIMBALANCE HANDLING COMPARISON (PR-AUC)")
print("=" * 40)
for method, score in sorted(results.items(), key=lambda x: x[1] or 0, reverse=True):
    print(f" {method:<20}: {score}")

## 1.2 Ensuring No Data Leakage

**Data leakage** = using future information to predict the future. Check for features with suspiciously high correlation with the target.

**Limitation:** Pearson correlation only detects linear relationships. Non-linear leakage (e.g., a feature that perfectly separates classes via thresholding) would be missed. For a more thorough check, inspect feature importance from a quick model fit and verify each top feature is causally valid.

In [ ]:
# Check for suspiciously high feature-target correlations
def detect_leakage(df, feature_cols, label_col, threshold=0.5):
    """Find features with suspiciously high target correlation (Pearson)."""
    suspicious = []
    y = df[label_col]

    for col in feature_cols:
        x = df[col]
        # Drop rows where either is NaN for valid correlation
        mask = x.notna() & y.notna()
        if mask.sum() < 10 or x[mask].std() == 0:
            continue
        corr = np.abs(np.corrcoef(x[mask], y[mask])[0, 1])
        if corr > threshold:
            suspicious.append((col, corr))

    return sorted(suspicious, key=lambda x: x[1], reverse=True)

suspicious_features = detect_leakage(df, feature_cols, label_col, threshold=0.3)

print("LEAKAGE DETECTION (Pearson correlation)")
print("=" * 50)
if suspicious_features:
    print(" Features with high target correlation:")
    for feat, corr in suspicious_features[:10]:
        print(f"  {feat}: {corr:.3f}")
else:
    print(" No obvious linear leakage detected")


**Interpretation**

If features like `hba1c_trend` or `current_egfr` appear above, they are **not** data leakage — they are legitimate clinical correlations (worsening HbA1c and declining kidney function genuinely predict adverse events).

However, strong correlations mean the model may become over-reliant on these features. If the data distribution shifts (e.g., a new patient population with different HbA1c patterns), model performance could degrade. We address this sensitivity in Session 4 (Robustness).

---

# Part 2: Hyperparameter Tuning

Now we tune model hyperparameters using the full dataset.

We'll build cross-validation and search functions from scratch to understand what happens under the hood, then use XGBoost's efficient built-in `xgb.cv()`.

## 2.1 Tuning Dataset

We use the same train/test split from Part 1 with patient-level cross-validation for hyperparameter search.


In [ ]:
# Same train/test split from Part 1
print(f"Tuning dataset: {len(X_train):,} instances from {groups_train.nunique()} patients")
print(f"Test holdout:   {len(X_test):,} instances")


## 2.2 Custom Group K-Fold Cross-Validation

Standard K-Fold CV randomly assigns instances to folds, but in healthcare one patient might appear in both train and validation — causing data leakage. **Group K-Fold** keeps all instances from one patient in the same fold.

We implement this from scratch to understand what's happening under the hood.

In [ ]:
def create_group_kfold_splits(groups, n_splits=3, random_state=42):
    """
    Create K-Fold splits that keep all instances from each group (patient) together.
    
    Parameters:
    -----------
    groups : array-like
        Group labels (patient IDs) for each instance
    n_splits : int
        Number of folds (K)
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    list of tuples: [(train_indices, val_indices), ...] for each fold
    """
    rng = np.random.RandomState(random_state)
    
    # Step 1: Get unique groups (patients) and shuffle them
    unique_groups = np.array(groups.unique())
    rng.shuffle(unique_groups)
    
    # Step 2: Split patients into K roughly equal parts
    group_folds = np.array_split(unique_groups, n_splits)
    
    # Step 3: Create train/val index splits
    splits = []
    groups_array = np.array(groups)
    
    for fold_idx in range(n_splits):
        # Validation patients = patients assigned to this fold
        val_groups = set(group_folds[fold_idx])
        
        # Find row indices for train and validation
        val_mask = np.array([g in val_groups for g in groups_array])
        train_indices = np.where(~val_mask)[0]
        val_indices = np.where(val_mask)[0]
        
        splits.append((train_indices, val_indices))
    
    return splits

print("create_group_kfold_splits() function defined")

In [ ]:
def cross_validate_model(model, X, y, groups, n_splits=3, scoring='pr_auc', random_state=42):
    """
    Perform group K-fold cross-validation on a model.
    
    Parameters:
    -----------
    model : sklearn estimator
        Model to evaluate (must have fit() and predict_proba() methods)
    X : DataFrame
        Feature matrix
    y : Series
        Target labels (0 or 1)
    groups : Series
        Patient IDs for grouping
    n_splits : int
        Number of CV folds
    scoring : str
        'pr_auc' for Precision-Recall AUC, 'roc_auc' for ROC AUC
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    dict with 'scores' (list of fold scores), 'mean', and 'std'
    """
    from sklearn.base import clone
    
    # Create the fold splits
    splits = create_group_kfold_splits(groups, n_splits, random_state)
    
    scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        # Get data for this fold
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_val_fold = y.iloc[val_idx]
        
        # Clone model to get a fresh instance
        model_clone = clone(model)
        
        # Train on this fold's training data
        model_clone.fit(X_train_fold, y_train_fold)
        
        # Get predicted probabilities for validation set
        y_pred_proba = model_clone.predict_proba(X_val_fold)[:, 1]
        
        # Calculate the appropriate score
        if scoring == 'pr_auc':
            score = average_precision_score(y_val_fold, y_pred_proba)
        else:  # roc_auc
            score = roc_auc_score(y_val_fold, y_pred_proba)
        
        scores.append(score)
    
    return {
        'scores': scores,
        'mean': np.mean(scores),
        'std': np.std(scores)
    }

print("cross_validate_model() function defined")


## 2.3 Random Forest Tuning

We implement a random hyperparameter search and use it to tune Random Forest.

In [ ]:
def random_hyperparameter_search(model_class, param_grid, X, y, groups, 
                                  n_iter=10, n_splits=3, scoring='pr_auc',
                                  random_state=42, verbose=True, **fixed_params):
    """
    Perform random hyperparameter search with group cross-validation.
    
    Parameters:
    -----------
    model_class : class
        Model class to instantiate (e.g., RandomForestClassifier)
    param_grid : dict
        Dictionary mapping parameter names to lists of values to try
        Example: {'max_depth': [5, 10, 15], 'n_estimators': [50, 100]}
    X, y, groups : DataFrames/Series
        Training data and patient groups
    n_iter : int
        Number of random combinations to try
    n_splits : int
        Number of CV folds for each evaluation
    scoring : str
        Metric to optimize ('pr_auc' or 'roc_auc')
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Whether to print progress
    **fixed_params : 
        Parameters to pass to all models (not tuned)
        
    Returns:
    --------
    dict with 'best_params', 'best_score', and 'all_results'
    """
    rng = np.random.RandomState(random_state)
    
    all_results = []
    best_score = -np.inf
    best_params = None
    
    for i in range(n_iter):
        # Randomly sample one value for each parameter
        params = {}
        for param_name, param_values in param_grid.items():
            params[param_name] = rng.choice(param_values)
        
        # Create model with sampled params + fixed params
        model = model_class(**params, **fixed_params)
        
        # Run cross-validation
        cv_results = cross_validate_model(
            model, X, y, groups, 
            n_splits=n_splits, 
            scoring=scoring, 
            random_state=random_state
        )
        
        # Store results
        result = {
            'params': params,
            'mean_score': cv_results['mean'],
            'std_score': cv_results['std'],
            'fold_scores': cv_results['scores']
        }
        all_results.append(result)
        
        # Update best if this is better
        if cv_results['mean'] > best_score:
            best_score = cv_results['mean']
            best_params = params.copy()
        
        if verbose:
            print(f"Iteration {i+1}/{n_iter}: CV {scoring}={cv_results['mean']:.4f} "
                  f"(+/- {cv_results['std']:.4f})")
    
    return {
        'best_params': best_params,
        'best_score': best_score,
        'all_results': all_results
    }

print("random_hyperparameter_search() function defined")

In [ ]:
# Define parameter grid and run search
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'class_weight': ['balanced', None]
}

print("Starting Random Forest hyperparameter search (20 iterations)...")
print("=" * 60)

rf_search_results = random_hyperparameter_search(
    model_class=RandomForestClassifier,
    param_grid=rf_param_grid,
    X=X_train, y=y_train, groups=groups_train,
    n_iter=20, n_splits=3, scoring='pr_auc',
    random_state=42, verbose=True,
    n_jobs=-1
)

print(f"\nBest RF params: {rf_search_results['best_params']}")
print(f"Best CV PR-AUC: {rf_search_results['best_score']:.4f}")


In [ ]:
# Visualize search results
scores = [r['mean_score'] for r in rf_search_results['all_results']]

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(scores)+1), scores, 'bo-', markersize=8)
plt.axhline(y=rf_search_results['best_score'], color='r', linestyle='--', 

    label=f'Best: {rf_search_results["best_score"]:.4f}')
plt.xlabel('Iteration')
plt.ylabel('CV PR-AUC')
plt.title('Random Forest Hyperparameter Search')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best RF on test set
best_rf = RandomForestClassifier(
    **rf_search_results['best_params'],
    random_state=42, n_jobs=-1
)
best_rf.fit(X_train, y_train)

y_pred_proba_rf = best_rf.predict_proba(X_test)[:, 1]
rf_roc_auc = roc_auc_score(y_test, y_pred_proba_rf)
rf_pr_auc = average_precision_score(y_test, y_pred_proba_rf)

print(f"Test Set Performance (Best Random Forest):")
print(f"  ROC-AUC: {rf_roc_auc:.4f}")
print(f"  PR-AUC:  {rf_pr_auc:.4f}")


## 2.4 XGBoost Hyperparameter Tuning

We use XGBoost's `xgb.cv()` combined with random hyperparameter sampling. For each randomly sampled configuration, `xgb.cv()` runs cross-validation with early stopping to automatically find the optimal number of boosting rounds.


---

<details>
<summary><strong>Hint 1 — xgb.cv workflow</strong> (click to expand)</summary>

```python
sampled = {k: rng.choice(v) for k, v in param_grid.items()}
params = {**base_params, **sampled}
cv_result = xgb.cv(params=params, dtrain=dtrain, num_boost_round=300,
                    folds=custom_folds, early_stopping_rounds=20,
                    verbose_eval=False, seed=42)
best_round = cv_result['test-aucpr-mean'].idxmax() + 1
```
</details>

<details>
<summary><strong>Hint 2 — Why xgb.cv over sklearn GridSearchCV?</strong> (click to expand)</summary>

`xgb.cv` supports early stopping natively — it stops adding trees when validation performance plateaus. This automatically determines the optimal `n_estimators` for each hyperparameter combination, which sklearn's CV cannot do.
</details>

In [ ]:
# Calculate scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / max(pos_count, 1)

print(f"Class imbalance ratio: {scale_pos_weight:.2f}")

# Create DMatrix from raw data (NaN passthrough for XGBoost)
dtrain = xgb.DMatrix(X_train_raw, label=y_train)
splits = create_group_kfold_splits(groups_train, n_splits=3, random_state=42)
custom_folds = [(ti.tolist(), vi.tolist()) for ti, vi in splits]

print(f"Created DMatrix with {dtrain.num_row():,} rows and {dtrain.num_col()} features")
print(f"Using NaN passthrough for XGBoost (native missing-value handling)")


In [ ]:
# XGBoost random hyperparameter search using xgb.cv() with early stopping
xgb_param_grid = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
}

base_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'scale_pos_weight': scale_pos_weight,
    'seed': 42
}

rng = np.random.RandomState(42)
n_iter = 15

# TODO: Implement the random search loop
#   For each iteration:
#     1. Sample one value per param: sampled = {k: rng.choice(v) for k, v in xgb_param_grid.items()}
#     2. Merge with base_params: params = {**base_params, **sampled}
#     3. Run xgb.cv(params=params, dtrain=dtrain, num_boost_round=300,
#                    folds=custom_folds, early_stopping_rounds=20, verbose_eval=False, seed=42)
#     4. Find best round: cv_result['test-aucpr-mean'].idxmax() + 1
#     5. Store result: {'params': sampled, 'best_round': ..., 'best_score': ...}
#
#   HINT: xgb.cv returns a DataFrame with 'test-aucpr-mean' column.
#         Use .idxmax() to find the best boosting round.
xgb_results = []
print(f"Running XGBoost random search ({n_iter} iterations with xgb.cv + early stopping)...")
print("=" * 70)

for i in range(n_iter):
    # TODO: Sample hyperparameters
    sampled = None
    params = None

    # TODO: Run xgb.cv with early stopping
    cv_result = None

    # TODO: Extract best round and score
    best_round = None
    best_score = None

    xgb_results.append({
        'params': sampled,
        'best_round': best_round,
        'best_score': best_score,
        'cv_history': cv_result
    })

    print(f"  [{i+1:2d}/{n_iter}] -> PR-AUC={best_score} (round {best_round})")

best_xgb_result = max(xgb_results, key=lambda x: x['best_score'] or 0)
print(f"\nBest: {best_xgb_result['params']}")
print(f"Best n_estimators: {best_xgb_result['best_round']}")
print(f"Best CV PR-AUC: {best_xgb_result['best_score']}")

### XGBoost Learning Curves

Learning curves show train vs validation performance over boosting rounds. The gap between curves indicates overfitting; early stopping finds the sweet spot.

In [ ]:
# Plot learning curves for the best configuration
cv_history = best_xgb_result['cv_history']

plt.figure(figsize=(12, 5))

plt.plot(cv_history['train-aucpr-mean'], label='Train', linewidth=2, color='blue')
plt.fill_between(
    range(len(cv_history)),
    cv_history['train-aucpr-mean'] - cv_history['train-aucpr-std'],
    cv_history['train-aucpr-mean'] + cv_history['train-aucpr-std'],
    alpha=0.2, color='blue'
)

plt.plot(cv_history['test-aucpr-mean'], label='Validation', linewidth=2, color='orange')
plt.fill_between(
    range(len(cv_history)),
    cv_history['test-aucpr-mean'] - cv_history['test-aucpr-std'],
    cv_history['test-aucpr-mean'] + cv_history['test-aucpr-std'],
    alpha=0.2, color='orange'
)

# best_round is 1-based (n_estimators), plot x-axis is 0-based
plt.axvline(x=best_xgb_result['best_round'] - 1, color='red', linestyle='--', 
    label=f'Best Round ({best_xgb_result["best_round"]})')

plt.xlabel('Boosting Round (Number of Trees)')
plt.ylabel('PR-AUC')
plt.title('XGBoost Learning Curves\n(Gap between curves indicates overfitting)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Train final XGBoost with best params on raw data (NaN passthrough)
# TODO: Build the final XGBClassifier using the best hyperparameters from the search
#   1. Extract params from best_xgb_result['params']
#   2. Create XGBClassifier with those params + scale_pos_weight + eval_metric='aucpr'
#   3. Fit on X_train_raw (not imputed!) with y_train
#   4. Evaluate on X_test_raw
#
#   HINT: Use best_xgb_result['best_round'] for n_estimators.
#         Cast params to correct types: int(params['max_depth']), float(params['learning_rate'])
best_xgb_params = best_xgb_result['params']

best_xgb = None  # TODO: Create XGBClassifier with best params

# TODO: Fit and evaluate
y_pred_proba_xgb = None
xgb_roc_auc = None
xgb_pr_auc = None

print(f"TUNED MODEL COMPARISON (Test Set)")
print("=" * 60)
print(f"{'Model':<35} {'ROC-AUC':>10} {'PR-AUC':>10}")
print("-" * 60)
print(f"{'Random Forest (tuned, median imp.)':<35} {rf_roc_auc:>10} {rf_pr_auc:>10}")
print(f"{'XGBoost (tuned, NaN passthrough)':<35} {xgb_roc_auc:>10} {xgb_pr_auc:>10}")
print("=" * 60)

### Save Best Hyperparameters

Save to JSON for use in the ensemble section below and in the homework.

In [ ]:
# Save best hyperparameters for use in homework
best_xgb_p = best_xgb_result['params']

params_to_save = {
    'random_forest': rf_search_results['best_params'],
    'xgboost': {
        'max_depth': int(best_xgb_p['max_depth']),
        'learning_rate': float(best_xgb_p['learning_rate']),
        'n_estimators': int(best_xgb_result['best_round']),
        'subsample': float(best_xgb_p['subsample']),
        'colsample_bytree': float(best_xgb_p['colsample_bytree']),
        'min_child_weight': int(best_xgb_p['min_child_weight'])
    },
    'xgboost_scale_pos_weight': float(scale_pos_weight)
}

# Convert numpy types for JSON
def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, (np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

hyperparams_path = os.path.join(OUTPUT_DIR, 'best_hyperparameters.json')
with open(hyperparams_path, 'w') as f:
    json.dump(convert_numpy(params_to_save), f, indent=2)

print(f"Saved to {hyperparams_path}")
print(json.dumps(convert_numpy(params_to_save), indent=2))

# Save the tuned XGBoost model for Session 3 and Session 4
# Note: this model was trained on raw data (NaN passthrough)
best_xgb.base_score = 0.5  # Ensure SHAP compatibility
xgb_model_path = os.path.join(OUTPUT_DIR, 'best_xgboost_model.pkl')
with open(xgb_model_path, 'wb') as f:
    pickle.dump(best_xgb, f)
print(f"Saved tuned XGBoost to {xgb_model_path}")

# Save the median imputer for downstream notebooks
imputer_path = os.path.join(OUTPUT_DIR, 'median_imputer.pkl')
with open(imputer_path, 'wb') as f:
    pickle.dump(imputer, f)
print(f"Saved median imputer to {imputer_path}")


---

# Part 3: Classifier Ensembles

Ensembles combine multiple models to improve prediction robustness. The idea: **different models make different errors**, so combining them can reduce overall error.

| Ensemble Type | How It Works | When to Use |
|---------------|--------------|-------------|
| **Voting** | Average predictions from all models | Simple baseline |
| **Weighted Voting** | Weight by model performance | When models differ in quality |
| **Stacking** | Meta-model learns to combine | Most sophisticated |

**Important caveat:** Ensembles can improve robustness, but gains over a strong single model like XGBoost are often marginal on structured/tabular data. The main benefit is reduced variance — the ensemble is less likely to have a bad day than any single model.

**Data note:** sklearn's `VotingClassifier` and `StackingClassifier` feed the same data to all estimators. This means XGBoost inside an ensemble receives median-imputed data (not raw NaN), losing its native missing-value handling. The standalone XGBoost in Part 2 uses NaN passthrough and may perform slightly differently.

## 3.1 Create Base Model Templates

We create base model templates using the hyperparameters tuned in Part 2. These templates are cloned (via `sklearn.base.clone`) for each ensemble method, so model parameters are defined in one place.


In [ ]:
# Base model templates from Part 2 tuning (unfitted)
rf_base = RandomForestClassifier(
    **rf_search_results['best_params'],
    random_state=42, n_jobs=-1
)

xgb_base = xgb.XGBClassifier(
    n_estimators=best_xgb_result['best_round'],
    max_depth=int(best_xgb_result['params']['max_depth']),
    learning_rate=float(best_xgb_result['params']['learning_rate']),
    subsample=float(best_xgb_result['params']['subsample']),
    colsample_bytree=float(best_xgb_result['params']['colsample_bytree']),
    min_child_weight=int(best_xgb_result['params']['min_child_weight']),
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='aucpr'
)

lr_base = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

print("Base model templates created from Part 2 tuning:")
print(f"  RF:  {rf_search_results['best_params']}")
print(f"  XGB: depth={int(best_xgb_result['params']['max_depth'])}, "
      f"lr={best_xgb_result['params']['learning_rate']}, "
      f"n_est={best_xgb_result['best_round']}")
print(f"  LR:  class_weight='balanced'")


## 3.2 Voting Ensembles

**Hard Voting**: Each model votes for a class, majority wins. 

**Soft Voting**: Average the probability predictions from all models (usually better).

**Note:** We use only RF and XGBoost as base estimators. Including Logistic Regression degrades ensemble performance because its PR-AUC (~0.31) is far below the tree models (~0.88). A weak model in a voting ensemble can pull down the average. Ensembles work best when base models are individually strong and make diverse errors.

In [ ]:
# Same train/test split from Part 1
print(f"Ensemble data: Train={len(X_train):,}, Test={len(X_test):,}")


In [ ]:
# Hard Voting (RF + XGBoost only — LR excluded due to poor individual performance)
from sklearn.metrics import accuracy_score, f1_score

# TODO: Create a VotingClassifier with 'hard' voting
#   estimators: [('rf', clone(rf_base)), ('xgb', clone(xgb_base))]
#   HINT: VotingClassifier(estimators=[...], voting='hard')
hard_voting = None

# TODO: Fit on training data and predict on test set
y_pred_hard = None

print(f"Hard Voting (RF + XGBoost):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_hard) if y_pred_hard is not None else None}")
print(f"  F1 Score: {f1_score(y_test, y_pred_hard) if y_pred_hard is not None else None}")

In [ ]:
# Soft Voting (RF + XGBoost only)
# TODO: Create a VotingClassifier with 'soft' voting
#   Same estimators as hard voting, but voting='soft' uses predict_proba
#   HINT: VotingClassifier(estimators=[...], voting='soft')
soft_voting = None

# TODO: Fit, get probabilities, compute ROC-AUC and PR-AUC
y_pred_soft = None
soft_roc = None
soft_pr = None

print(f"Soft Voting (RF + XGBoost):")
print(f"  ROC-AUC: {soft_roc}")
print(f"  PR-AUC:  {soft_pr}")

## 3.3 Weighted Voting

Weight each model by its individual performance. Better models get more influence.

**Note:** When base models have nearly identical performance (RF PR-AUC ~0.876, XGB PR-AUC ~0.877), the weights degenerate to approximately equal (0.50/0.50). Weighted voting only meaningfully differs from soft voting when base models have clearly different performance levels.

---

<details>
<summary><strong>Hint 1 — Weighted averaging</strong> (click to expand)</summary>

Weight each model's predictions proportionally to its individual PR-AUC:
```python
total = rf_pr + xgb_pr
weights = [rf_pr/total, xgb_pr/total]
weighted_pred = weights[0]*rf_pred + weights[1]*xgb_pred
```
</details>

In [ ]:
# Fit individual models for comparison and weighted voting
# TODO: Fit each base model individually and get their predictions
#   1. Clone and fit rf_base, xgb_base, lr_base on training data
#   2. Get predict_proba on test set for each
#   3. Compute PR-AUC for each
rf_fitted = None
xgb_fitted = None
lr_fitted = None

rf_pred = None
xgb_pred = None
lr_pred = None

rf_pr = None
xgb_pr = None
lr_pr = None

print("Individual Model Performance (PR-AUC):")
print(f"  Random Forest:  {rf_pr}")
print(f"  XGBoost:        {xgb_pr}")
print(f"  Logistic Reg:   {lr_pr}")

# TODO: Create performance-based weights for RF + XGBoost only
#   weight_i = pr_auc_i / sum(pr_auc_values)
#   Then compute weighted average: weights[0]*rf_pred + weights[1]*xgb_pred
#   HINT: total_pr = rf_pr + xgb_pr, then [rf_pr/total_pr, xgb_pr/total_pr]
weights = None
weighted_pred = None
weighted_roc = None
weighted_pr = None

print(f"\nWeighted Ensemble: ROC-AUC={weighted_roc}, PR-AUC={weighted_pr}")

## 3.4 Stacking Classifier

Stacking trains a **meta-model** to learn how to best combine the base model predictions. It uses cross-validation internally to generate meta-features, avoiding overfitting.

---

<details>
<summary><strong>Hint 1 — StackingClassifier API</strong> (click to expand)</summary>

```python
StackingClassifier(
    estimators=[('rf', clone(rf_base)), ('xgb', clone(xgb_base))],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=stack_cv_splits,
    passthrough=False
)
```

The `cv` parameter accepts pre-computed splits — use the GroupKFold splits to ensure patient-level separation in internal cross-validation.
</details>

In [ ]:
# Stacking classifier with patient-aware internal CV
# TODO: Create a StackingClassifier
#   - Base estimators: [('rf', clone(rf_base)), ('xgb', clone(xgb_base))]
#   - Meta-learner (final_estimator): LogisticRegression(max_iter=1000)
#   - cv: Use patient-aware splits (gkf_stack splits below)
#   - passthrough=False (only pass base model predictions to meta-learner)
#
#   HINT: GroupKFold(n_splits=3) for patient-aware CV
print("Training stacking classifier...")

# NOTE: sklearn's GroupKFold assigns patients to folds deterministically
# (sorted order), unlike our custom create_group_kfold_splits() which
# shuffles patients first. Both are valid — deterministic assignment is
# fine here since StackingClassifier only needs consistent internal splits.
gkf_stack = GroupKFold(n_splits=3)
stack_cv_splits = list(gkf_stack.split(X_train, y_train, groups=groups_train))

stacking = None  # TODO: Create StackingClassifier

# TODO: Fit and evaluate
y_pred_stack = None
stack_roc = None
stack_pr = None

print(f"Stacking: ROC-AUC={stack_roc}, PR-AUC={stack_pr}")

## 3.5 Final Ensemble Comparison

In [ ]:
print("\n" + "=" * 60)
print("ENSEMBLE COMPARISON")
print("=" * 60)
print(f"{'Method':<25} {'ROC-AUC':>12} {'PR-AUC':>12}")
print("-" * 60)
print(f"{'Random Forest (single)':<25} {roc_auc_score(y_test, rf_pred):>12.4f} {rf_pr:>12.4f}")
print(f"{'XGBoost (single)':<25} {roc_auc_score(y_test, xgb_pred):>12.4f} {xgb_pr:>12.4f}")
print(f"{'Logistic Reg (single)':<25} {roc_auc_score(y_test, lr_pred):>12.4f} {lr_pr:>12.4f}")
print("-" * 60)
print(f"{'Soft Voting':<25} {soft_roc:>12.4f} {soft_pr:>12.4f}")
print(f"{'Weighted Voting':<25} {weighted_roc:>12.4f} {weighted_pr:>12.4f}")
print(f"{'Stacking':<25} {stack_roc:>12.4f} {stack_pr:>12.4f}")
print("=" * 60)


### Save Best Ensemble Model

In [ ]:
# Auto-select and save the best ensemble model
ensemble_candidates = {
    'soft_voting': (soft_voting, soft_pr),
    'stacking': (stacking, stack_pr),
}

best_name, (best_ensemble, best_pr) = max(
    ensemble_candidates.items(), key=lambda x: x[1][1]
)

ensemble_path = os.path.join(OUTPUT_DIR, 'best_ensemble_model.pkl')
with open(ensemble_path, 'wb') as f:
    pickle.dump(best_ensemble, f)

print(f"Best ensemble: {best_name} (PR-AUC={best_pr:.4f})")
print(f"Saved to {ensemble_path}")


---

## Summary: Session Complete!

### Part 1: Advanced Dataset Creation

1. **Imbalance**: Class weights, SMOTE, or undersampling to handle rare events
2. **Leakage**: Check feature correlations, always use GroupKFold/GroupShuffleSplit

### Part 2: Hyperparameter Tuning

1. **Custom Group K-Fold CV**: Patient-aware cross-validation from scratch
2. **Random Search**: Custom implementation for Random Forest tuning
3. **XGBoost xgb.cv()**: Built-in CV with learning curves and early stopping
4. **Saved hyperparameters** to `data/week_3/best_hyperparameters.json`

### Part 3: Classifier Ensembles

1. **Voting**: Simple average of predictions (soft voting usually better)
2. **Weighted Voting**: Weight models by their performance
3. **Stacking**: Meta-model learns to combine base predictions
4. **Saved best ensemble** to `data/week_3/best_ensemble_model.pkl`

### Key Takeaways

- Dataset creation is as important as model selection
- Always use patient-level cross-validation for healthcare data
- Ensembles improve robustness but gains over a strong single XGBoost are marginal on this dataset — the main benefit is reduced variance, not higher peak performance

### Next Steps

In **Session 3**, you'll learn about model explainability using SHAP to understand what drives predictions.

### Professional Tip

In production, monitor individual model performance within your ensemble. If one model starts degrading (due to data drift), you can quickly identify and retrain just that component.